# Analyse exploratoire des données — Prédiction du churn télécom

**Projet :** Industrialisation MLOps d’un modèle de prédiction du churn  
**Objectif :** examiner la structure, la qualité et les relations entre variables du jeu de données avant l’entraînement et le déploiement du modèle.

> Les données utilisées dans ce TP sont **synthétiques**. Les constats obtenus ne doivent pas être généralisés à une clientèle réelle sans validation complémentaire.


## 1. Bibliothèques et configuration

Les chemins sont résolus automatiquement, que le notebook soit exécuté depuis la racine du projet ou depuis le dossier `notebooks`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (cwd, *cwd.parents) if (p / "data" / "raw" / "churn.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Dataset introuvable : ouvrir le projet ml-churn-project ou son dossier notebooks dans VS Code."
    )
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "churn.csv"
FIG_DIR = PROJECT_ROOT / "data"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f"Projet : {PROJECT_ROOT}")
print(f"Dataset : {DATA_PATH}")


## 2. Chargement et aperçu du dataset

La variable cible `churn` indique si un client a quitté le service (`1`) ou non (`0`).

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Dimensions : {df.shape[0]} lignes × {df.shape[1]} colonnes")
display(df.head(10))


## 3. Dictionnaire des variables

Les variables suivantes correspondent au schéma d’entrée de l’API FastAPI du projet. Les unités exactes dépendent des conventions du générateur de données.

In [ ]:
descriptions = {
    "age": "Âge du client",
    "anciennete": "Ancienneté du client",
    "revenu": "Revenu du client",
    "nb_reclamations": "Nombre de réclamations",
    "montant_mensuel": "Montant mensuel facturé",
    "retards_paiement": "Nombre de retards de paiement",
    "churn": "Variable cible : 1 = départ, 0 = maintien",
}
dictionary = pd.DataFrame([
    {"Variable": col, "Type": str(df[col].dtype), "Description": descriptions.get(col, "Variable complémentaire")}
    for col in df.columns
])
display(dictionary)


## 4. Contrôle de la qualité des données

Nous examinons les types, les valeurs manquantes et les doublons. La présence de doublons doit être interprétée selon la structure du dataset (un identifiant client serait nécessaire pour vérifier les doublons métier).

In [ ]:
quality = pd.DataFrame({
    "type": df.dtypes.astype(str),
    "valeurs_manquantes": df.isna().sum(),
    "taux_manquant_%": (df.isna().mean() * 100).round(2),
    "valeurs_uniques": df.nunique(dropna=True),
})
display(quality)
print(f"Lignes entièrement dupliquées : {df.duplicated().sum()}")
print(f"Total de valeurs manquantes : {int(df.isna().sum().sum())}")


## 5. Statistiques descriptives

Les statistiques descriptives permettent de repérer les ordres de grandeur, la dispersion et d’éventuelles valeurs atypiques.

In [ ]:
display(df.describe(include="all").T)


## 6. Répartition de la variable cible `churn`

Nous calculons les effectifs et proportions des deux classes. Une répartition déséquilibrée peut influencer le choix des métriques d’évaluation du modèle.

In [ ]:
if "churn" not in df.columns:
    raise KeyError("La colonne cible 'churn' est absente du dataset.")
counts = df["churn"].value_counts(dropna=False).sort_index()
percentages = (df["churn"].value_counts(normalize=True, dropna=False).sort_index() * 100).round(2)
display(pd.DataFrame({"effectif": counts, "pourcentage_%": percentages}))
print(f"Taux de churn : {df['churn'].eq(1).mean() * 100:.2f} %")

fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=df, x="churn", order=[0, 1], ax=ax)
ax.set(title="Répartition des clients selon le churn", xlabel="Churn (0 = maintien, 1 = départ)", ylabel="Nombre de clients")
for container in ax.containers:
    ax.bar_label(container)
fig.tight_layout()
fig.savefig(FIG_DIR / "churn_distribution.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. Distribution des variables explicatives

Nous visualisons les distributions des variables numériques afin de repérer asymétries et valeurs extrêmes. La variable cible est exclue de ces histogrammes.

In [ ]:
features = df.select_dtypes(include="number").drop(columns=["churn"], errors="ignore")
if features.empty:
    print("Aucune variable explicative numérique à représenter.")
else:
    axes = features.hist(figsize=(12, 8), bins=25, edgecolor="white")
    plt.suptitle("Distribution des variables numériques", y=1.02)
    plt.tight_layout()
    plt.show()


## 8. Corrélations entre variables numériques

La corrélation de Pearson décrit une **association linéaire** entre deux variables. Elle ne prouve pas un lien de causalité. Pour une cible binaire codée 0/1, les coefficients donnent une première indication descriptive.

In [ ]:
numeric = df.select_dtypes(include="number")
if numeric.shape[1] < 2:
    print("Pas assez de variables numériques pour calculer une matrice de corrélation.")
else:
    corr = numeric.corr()
    display(corr.round(3))
    fig, ax = plt.subplots(figsize=(10, 7))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
    ax.set_title("Matrice de corrélation (Pearson)")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "correlation.png", dpi=150, bbox_inches="tight")
    plt.show()
    if "churn" in corr.columns:
        print("Corrélations avec churn (ordre décroissant en valeur absolue) :")
        display(corr["churn"].drop("churn").reindex(corr["churn"].drop("churn").abs().sort_values(ascending=False).index).to_frame("correlation_avec_churn"))


## 9. Comparaison descriptive selon le churn

Cette comparaison met en évidence les moyennes des variables numériques par classe. Les écarts observés ne démontrent pas, à eux seuls, qu’une variable prédit correctement le churn.

In [ ]:
numeric_features = df.select_dtypes(include="number").drop(columns=["churn"], errors="ignore").columns
if len(numeric_features) > 0:
    display(df.groupby("churn")[list(numeric_features)].mean().T.round(2).rename(columns={0: "Clients maintenus", 1: "Clients partis"}))
else:
    print("Aucune variable numérique disponible pour cette comparaison.")


## 10. Synthèse et implications pour la modélisation

- **Qualité des données :** consulter les valeurs manquantes, doublons et types identifiés ci-dessus avant l’entraînement.
- **Répartition de la cible :** interpréter le taux de churn affiché et choisir des métriques adaptées (précision, rappel, F1, matrice de confusion), en complément de l’accuracy.
- **Relations entre variables :** utiliser les corrélations et comparaisons par groupe comme outils exploratoires, sans conclure à la causalité.
- **Industrialisation :** le projet entraîne et sauvegarde un modèle dans `models/model.pkl`, puis l’expose via FastAPI. Les évaluations comparatives des modèles sont réalisées dans `src/train.py`.

**Limites :** données synthétiques, analyse descriptive, absence d’inférence causale et nécessité d’une validation sur des données indépendantes avant tout usage opérationnel.
